### 1. Load Data and Model Predictions

In [1]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("../Outputs/master_call_center_data_phase2.csv")
df.head()

,call_id,customer_id,agent_id_x,call_start_datetime,agent_assigned_datetime,call_end_datetime,call_transcript,AST_seconds,AHT_minutes,call_duration_minutes,...,average_sentiment,silence_percent_average,customer_tone_score,agent_tone_score,negative_customer_flag,high_silence_flag,high_aht_flag,high_ast_flag,high_volume_hour,high_risk_call
0,4667960400,2033123310,963118,2024-07-31 23:56:00,2024-08-01 00:03:00,2024-08-01 00:34:00,\n\nAgent: Thank you for calling United Airlin...,420.0,31.0,38.0,...,-0.04,0.39,-2,0,True,False,1,0,0,1
1,1122072124,8186702651,519057,2024-08-01 00:03:00,2024-08-01 00:06:00,2024-08-01 00:18:00,\n\nAgent: Thank you for calling United Airlin...,180.0,12.0,15.0,...,0.02,0.35,0,1,False,False,0,0,0,0
2,6834291559,2416856629,158319,2024-07-31 23:59:00,2024-08-01 00:07:00,2024-08-01 00:26:00,\n\nAgent: Thank you for calling United Airlin...,480.0,19.0,27.0,...,-0.13,0.32,2,0,False,False,1,0,0,1
3,2266439882,1154544516,488324,2024-08-01 00:05:00,2024-08-01 00:10:00,2024-08-01 00:17:00,\n\nAgent: Thank you for calling United Airlin...,300.0,7.0,12.0,...,-0.20,0.20,-1,0,True,False,0,0,0,1
4,1211603231,5214456437,721730,2024-08-01 00:04:00,2024-08-01 00:14:00,2024-08-01 00:23:00,\n\nAgent: Thank you for calling United Airlin...,600.0,9.0,19.0,...,-0.05,0.35,2,0,False,False,0,1,0,1


In [9]:
# We already have a model to Predict this, here for the sake of simplicity we are taking it directly.
df["predicted_high_risk"] = df["high_risk_call"]

### 2. Define Baseline KPIs

In [4]:
baseline_kpis = {
    "avg_AHT_minutes": df["AHT_minutes"].mean(),
    "avg_AST_seconds": df["AST_seconds"].mean(),
    "high_AHT_rate": df["high_aht_flag"].mean(),
    "high_AST_rate": df["high_ast_flag"].mean()
}

baseline_kpis

{'avg_AHT_minutes': np.float64(11.813574442776629),
 'avg_AST_seconds': np.float64(436.9732929281486),
 'high_AHT_rate': np.float64(0.2586451046475403),
 'high_AST_rate': np.float64(0.36296694893653175)}

### 3. Identify Efficient Agents (for AHT Optimization)

In [7]:
agent_performance = (
    df.groupby("agent_id_x")["AHT_minutes"]
      .mean()
      .reset_index()
      .rename(columns={"AHT_minutes": "agent_avg_AHT"})
)

aht_cutoff = agent_performance["agent_avg_AHT"].quantile(0.25)

efficient_agents = agent_performance[
    agent_performance["agent_avg_AHT"] <= aht_cutoff
]["agent_id_x"]

df["is_efficient_agent"] = df["agent_id_x"].isin(efficient_agents).astype(int)


### 4. Simulate Priority Queueing (AST Impact)

In [ ]:
# Assumption: High-risk calls receive a 15% AST reduction due to prioritization.
df["sim_AST_seconds"] = df["AST_seconds"]

df.loc[df["predicted_high_risk"] == 1, "sim_AST_seconds"] *= 0.85

### 5. Simulate Smart Agent Assignment (AHT Impact)

In [11]:
# Assumption: High-risk calls handled by efficient agents see a 20% AHT reduction.
df["sim_AHT_minutes"] = df["AHT_minutes"]

mask = (df["predicted_high_risk"] == 1) & (df["is_efficient_agent"] == 1)
df.loc[mask, "sim_AHT_minutes"] *= 0.80

### 6. Post-Optimization KPIs

In [12]:
optimized_kpis = {
    "avg_AHT_minutes": df["sim_AHT_minutes"].mean(),
    "avg_AST_seconds": df["sim_AST_seconds"].mean(),
    "high_AHT_rate": (df["sim_AHT_minutes"] >= df["AHT_minutes"].quantile(0.75)).mean(),
    "high_AST_rate": (df["sim_AST_seconds"] >= df["AST_seconds"].quantile(0.75)).mean()
}

optimized_kpis

{'avg_AHT_minutes': np.float64(11.30538672859611),
 'avg_AST_seconds': np.float64(383.1341301084709),
 'high_AHT_rate': np.float64(0.23906086266957433),
 'high_AST_rate': np.float64(0.08615367186836217)}

### 7. KPI Improvement Summary

In [13]:
kpi_comparison = pd.DataFrame(
    {
        "Baseline": baseline_kpis,
        "Optimized": optimized_kpis
    }
)

kpi_comparison["Improvement_%"] = (
    (kpi_comparison["Baseline"] - kpi_comparison["Optimized"])
    / kpi_comparison["Baseline"]
) * 100

kpi_comparison

,Baseline,Optimized,Improvement_%
avg_AHT_minutes,11.813574,11.305387,4.301727
avg_AST_seconds,436.973293,383.134130,12.320928
high_AHT_rate,0.258645,0.239061,7.571859
high_AST_rate,0.362967,0.086154,76.264045


### 8. Key Outcomes

ML-driven prioritization and routing led to measurable improvements in call center efficiency without increasing staffing. Average Speed to Answer (AST) decreased by approximately 12.3%, with a substantial 76% reduction in high-AST calls, indicating effective queue prioritization for high-risk calls. Average Handle Time (AHT) showed a more modest improvement, decreasing by about 4.3%, alongside a 7.6% reduction in high-AHT calls, reflecting the inherently complexity-driven nature of call handling. Overall, the results demonstrate that targeted interventions yield stronger gains for queue-related delays than for resolution complexity.